In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc, accuracy_score)
from sklearn.preprocessing import label_binarize
from matplotlib.backends.backend_pdf import PdfPages

# === CONFIGURACIÓN ===
INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected"
BASE_OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\models"
MODEL_NAME = "LightGBM"
INTENTO = 1

# Crear subcarpeta de salida
fecha_actual = datetime.today().strftime('%Y-%m-%d')
OUTPUT_PATH = os.path.join(BASE_OUTPUT_PATH, f"{MODEL_NAME}_{INTENTO:02d}_{fecha_actual}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# === PARÁMETROS ===
param_grid = {
    'num_leaves': [15, 31],
    'max_depth': [5, 10, -1],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 200]
}
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# === EJECUCIÓN ===
metricas_totales = []
mejor_modelo = None
mejor_score = -1

variantes_X = sorted([f for f in os.listdir(INPUT_PATH) if f.startswith("X_train")])

for x_file in variantes_X:
    base_name = x_file.replace("X_train_", "").replace(".parquet", "")
    try:
        print(f"\n🔍 Procesando: {base_name}")

        # Cargar datos
        X_train = pd.read_parquet(os.path.join(INPUT_PATH, f"X_train_{base_name}.parquet"))
        X_test = pd.read_parquet(os.path.join(INPUT_PATH, f"X_test_{base_name}.parquet"))
        y_train = pd.read_parquet(os.path.join(INPUT_PATH, f"y_train_{base_name}.parquet")).squeeze()
        y_test = pd.read_parquet(os.path.join(INPUT_PATH, f"y_test_{base_name}.parquet")).squeeze()

        # GridSearch con CV
        start = time.time()
        clf = LGBMClassifier(class_weight='balanced', random_state=42)
        grid = GridSearchCV(clf, param_grid, cv=cv_strategy, scoring='f1_weighted', n_jobs=-1)
        grid.fit(X_train, y_train)
        tiempo = (time.time() - start) / 60

        best_model = grid.best_estimator_

        # Predicciones
        y_pred = best_model.predict(X_test)
        y_proba = best_model.predict_proba(X_test)
        y_pred_train = best_model.predict(X_train)
        y_proba_train = best_model.predict_proba(X_train)
        y_pred_cv = cross_val_predict(best_model, X_train, y_train, cv=cv_strategy, method='predict')

        # Reporte
        report_test = pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).transpose()
        report_train = pd.DataFrame(classification_report(y_train, y_pred_train, output_dict=True)).transpose()
        report_cv = pd.DataFrame(classification_report(y_train, y_pred_cv, output_dict=True)).transpose()

        report_test["set"] = "test"
        report_train["set"] = "train"
        report_cv["set"] = "cv"

        full_report = pd.concat([report_train, report_cv, report_test])
        full_report.to_csv(os.path.join(OUTPUT_PATH, f"metricas_{base_name}.csv"))

        # Guardar PDF
        with PdfPages(os.path.join(OUTPUT_PATH, f"reporte_{base_name}.pdf")) as pdf:
            # Confusion Matrix
            cm = confusion_matrix(y_test, y_pred)
            ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')
            plt.title("Matriz de Confusión (Test)")
            pdf.savefig(); plt.close()

            # ROC
            classes = np.unique(y_test)
            y_bin = label_binarize(y_test, classes=classes)
            plt.figure()
            for i, clase in enumerate(classes):
                fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
                roc_auc = auc(fpr, tpr)
                plt.plot(fpr, tpr, label=f"Clase {clase} (AUC={roc_auc:.2f})")
            plt.plot([0, 1], [0, 1], 'k--')
            plt.title("Curvas ROC (Test)")
            plt.xlabel("FPR")
            plt.ylabel("TPR")
            plt.legend()
            pdf.savefig(); plt.close()

        # Resumen
        f1_test = report_test.loc['1', 'f1-score'] if '1' in report_test.index else 0
        f1_train = report_train.loc['1', 'f1-score'] if '1' in report_train.index else 0
        f1_cv = report_cv.loc['1', 'f1-score'] if '1' in report_cv.index else 0

        metricas_totales.append({
            "modelo": base_name,
            "f1_train": f1_train,
            "f1_cv": f1_cv,
            "f1_test": f1_test,
            "f1_train_minus_test": f1_train - f1_test,
            "f1_cv_minus_test": f1_cv - f1_test,
            "tiempo_min": tiempo
        })

        print(f"✅ {base_name}: F1_test_clase_1={f1_test:.3f}, F1_cv={f1_cv:.3f}, Tiempo={tiempo:.2f}min")

    except Exception as e:
        print(f"❌ Error en {base_name}: {e}")

# Guardar resumen final
pd.DataFrame(metricas_totales).to_csv(os.path.join(OUTPUT_PATH, "resumen_modelos_lightgbm.csv"), index=False)



🔍 Procesando: MaxAbs_FE_ALL
